# Intrusion Detection System — Modelling (Part 1: Binary Classification)
### Dataset: CICIDS-2017  ·  Runs on **Kaggle**

Runs *after* `FeatureSelection.ipynb`. First model: **Logistic Regression implemented from scratch in NumPy** — this is scratch-model #1 of the project's *"at least 2 from-scratch"* requirement.

**This notebook:**
| # | Section | Notes |
|---|---------|-------|
| 1 | **Data Journey** | Visualise the whole pipeline so far — rows & features at every stage (raw → preprocessing → FE → FS) |
| 2 | Load data | both training strategies: real-distribution + SMOTE-balanced; shared real-world test set |
| 3 | Logistic Regression **from scratch** | sigmoid, binary cross-entropy, **mini-batch gradient descent**, L2 regularisation — no `sklearn` model |
| 4 | Hyperparameter tuning | **grid search** over learning rate × L2 λ × batch size, with loss curves + a results heatmap |
| 5 | **SMOTE vs class-weighting** | train the best config **both ways**, compare head-to-head on the test set |
| 6 | Final evaluation | winning strategy — accuracy, precision, recall, F1, ROC-AUC, confusion matrix, learned weights |
| 7 | Save | both trained models + comparison table + metrics + figures |

**Task:** binary — `label_binary` (0 = BENIGN, 1 = ATTACK). Multi-class is a later notebook.

**The imbalance question:** the data is ~83% BENIGN. We compare two ways of handling that — **(A) class weighting** on the real data vs **(B) SMOTE** synthetic balancing — and let the test set decide which wins.

**Kaggle setup:** attach the `FeatureSelection.ipynb` output (containing `train_selected.parquet`, `train_binary_smote_selected.parquet`, `test_selected.parquet`, `selected_features.json`) as an input dataset, then set `IN_DIR` below to its mount path. No `pip install` needed for this notebook.

## 1. Imports, Config & Report Helpers

In [ ]:
import os, json, time, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             precision_recall_curve, classification_report)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.bbox'] = 'tight'

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Kaggle paths ───────────────────────────────────────────────────
# IN_DIR : the Kaggle Dataset / Notebook-Output holding FeatureSelection.ipynb results
#          (train_selected.parquet / test_selected.parquet / selected_features.json).
#   ----> EDIT to match the mount path shown in the Kaggle 'Input' panel on the right.
IN_DIR      = '/kaggle/input/ids-selected'
OUT_DIR     = '/kaggle/working'
FIGURES_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── report helpers ─────────────────────────────────────────────────
_report_lines = []

def _log(text=''):
    _report_lines.append(str(text))
    print(text)

def _savefig(name, fig=None):
    path = os.path.join(FIGURES_DIR, name)
    (fig or plt).savefig(path, dpi=130, bbox_inches='tight')
    return path

def write_report():
    path = os.path.join(OUT_DIR, 'Modeling_Binary_Report.txt')
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(_report_lines))
    print(f'\nReport saved -> {path}')

_log('=' * 70)
_log('MODELLING REPORT (BINARY)  —  CICIDS-2017')
_log('Model : Logistic Regression (from scratch, NumPy)')
_log(f'Generated : {datetime.now().strftime("%Y-%m-%d %H:%M")}')
_log('=' * 70)
print('\nSetup complete.')
print('  Reading from :', IN_DIR)
print('  Writing to   :', OUT_DIR)

## 2. The Data Journey — From Raw to Model-Ready
Before modelling, a visual recap of **where we started and where we are now**. Every stage of the pipeline changed the data — this section shows the row count and feature count at each step in one place.

*(These numbers are recorded from the reports of the earlier notebooks. The live-loaded shapes are verified against them in Section 3.)*

In [ ]:
# pipeline stages — (stage, rows, n_features, note)
JOURNEY = [
    ('Raw CICIDS-2017',      2830743, 79, '8 CSVs concatenated'),
    ('After Preprocessing',  2574264, 49, '-256k duplicates, -8 zero-var, -21 correlated cols'),
    ('Train split (80%)',    2059411, 49, 'stratified split; test = 514,853 rows'),
    ('After Feature Eng.',   2059411, 58, '+2 Init_Win flags, +7 derived features'),
    ('After Feature Select', 2059411, 47, '-11 redundant features (|r|>0.95)'),
    ('SMOTE-balanced train', 3437418, 47, 'binary task: ATTACK oversampled to match BENIGN'),
]
journey_df = pd.DataFrame(JOURNEY, columns=['stage', 'rows', 'features', 'note'])

_log('')
_log('── SECTION 1 : DATA JOURNEY ───────────────────────────────')
_log(journey_df.to_string(index=False))
display(journey_df)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 12))
stages = journey_df['stage']
colors = sns.color_palette('viridis', len(journey_df))

# ── (a) ROW COUNT funnel across stages ──
ax = axes[0, 0]
bars = ax.barh(range(len(journey_df)), journey_df['rows'], color=colors)
ax.set_yticks(range(len(journey_df)))
ax.set_yticklabels(stages)
ax.invert_yaxis()
ax.set_xlabel('Row count')
ax.set_title('Row Count at Each Pipeline Stage', fontsize=12, fontweight='bold')
for i, v in enumerate(journey_df['rows']):
    ax.text(v, i, f'  {v:,}', va='center', fontsize=9)

# ── (b) FEATURE COUNT flow ──
ax = axes[0, 1]
ax.plot(range(len(journey_df)), journey_df['features'], 'o-', color='#D32F2F',
        linewidth=2.5, markersize=10)
for i, v in enumerate(journey_df['features']):
    ax.annotate(f'{v}', (i, v), textcoords='offset points', xytext=(0, 12),
                ha='center', fontsize=11, fontweight='bold')
ax.set_xticks(range(len(journey_df)))
ax.set_xticklabels(stages, rotation=25, ha='right')
ax.set_ylabel('Number of features')
ax.set_title('Feature Count Through the Pipeline', fontsize=12, fontweight='bold')
ax.set_ylim(0, max(journey_df['features']) * 1.25)

# ── (c) feature engineering vs reduction — what was added / removed ──
ax = axes[1, 0]
fe_steps  = ['Raw\nfeatures', 'Preproc\ndrops', 'FE\nadditions', 'FS\ndrops', 'Final']
fe_deltas = [79, -30, +9, -11, 47]
running = [79]
for d in fe_deltas[1:-1]:
    running.append(running[-1] + d)
running.append(47)
bar_colors = ['#1976D2', '#F44336', '#4CAF50', '#F44336', '#1976D2']
ax.bar(fe_steps, [79, 30, 9, 11, 47], color=bar_colors)
for i, (lbl, val) in enumerate(zip(['79', '-30', '+9', '-11', '47'], [79, 30, 9, 11, 47])):
    ax.text(i, val, lbl, ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('feature count / change')
ax.set_title('Features: Added (green) vs Removed (red)', fontsize=12, fontweight='bold')
ax.legend(handles=[mpatches.Patch(color='#1976D2', label='count'),
                   mpatches.Patch(color='#4CAF50', label='added'),
                   mpatches.Patch(color='#F44336', label='removed')])

# ── (d) before vs now summary ──
ax = axes[1, 1]
ax.axis('off')
summary = (
    'WHERE WE STARTED                WHERE WE ARE NOW\n'
    '─────────────────────           ─────────────────────\n'
    f'Rows      2,830,743             Train  2,059,411 (real)\n'
    f'                                + 3,437,418 (SMOTE)\n'
    f'Features  79 raw                47 selected, scaled\n'
    f'Classes   15 (imbalanced)       7 families / binary\n'
    f'Quality   dupes, Inf, NaN,      clean, scaled,\n'
    f'          skewed, redundant     balanced, non-redundant\n'
    '\n'
    'PIPELINE:\n'
    '  Raw -> EDA -> Preprocessing -> Feature Eng.\n'
    '      -> Feature Selection -> [MODELLING]'
)
ax.text(0.0, 0.95, summary, fontsize=11, family='monospace', va='top',
        transform=ax.transAxes)
ax.set_title('Raw  vs  Model-Ready', fontsize=12, fontweight='bold')

plt.suptitle('THE DATA JOURNEY — CICIDS-2017 Intrusion Detection Pipeline',
             fontsize=15, fontweight='bold')
plt.tight_layout()
_savefig('01_data_journey.png', fig)
plt.show()

## 3. Load Selected Data — Both Imbalance Strategies
The training set is ~83% BENIGN. There are two standard ways to stop a model just predicting BENIGN:
- **(A) Class weighting** — train on the real-distribution data, but weight the loss by inverse class frequency. No synthetic data.
- **(B) SMOTE** — train on a synthetically balanced 50/50 dataset (`train_binary_smote_selected.parquet`).

This notebook does **both** and compares them head-to-head. We load:
- `train_selected.parquet` — real distribution, 47 features → used for strategy A and for hyperparameter tuning
- `train_binary_smote_selected.parquet` — SMOTE-balanced, 47 features → used for strategy B
- `test_selected.parquet` — real-world distribution, untouched, for honest evaluation of **both**

A **validation split** is carved from the real training data for hyperparameter tuning, so the test set is touched exactly once at the end.

In [ ]:
train_path = os.path.join(IN_DIR, 'train_selected.parquet')
smote_path = os.path.join(IN_DIR, 'train_binary_smote_selected.parquet')
test_path  = os.path.join(IN_DIR, 'test_selected.parquet')
feat_path  = os.path.join(IN_DIR, 'selected_features.json')

for p in [train_path, smote_path, test_path, feat_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f'{p} not found. Set IN_DIR to the Kaggle mount path of your '
            'FeatureSelection output (see the Input panel on the right). '
            'It must contain train_selected, train_binary_smote_selected, '
            'test_selected (.parquet) and selected_features.json.')

with open(feat_path) as f:
    selected_features = json.load(f)['selected_features']

train_df = pd.read_parquet(train_path)
smote_df = pd.read_parquet(smote_path)
test_df  = pd.read_parquet(test_path)

# ── strategy A inputs: real-distribution training data ──
X_train_full = train_df[selected_features].values.astype(np.float64)
y_train_full = train_df['label_binary'].values.astype(np.float64)

# ── strategy B inputs: SMOTE-balanced training data ──
X_smote = smote_df[selected_features].values.astype(np.float64)
y_smote = smote_df['label_binary'].values.astype(np.float64)

# ── shared test set (real-world distribution) ──
X_test = test_df[selected_features].values.astype(np.float64)
y_test = test_df['label_binary'].values.astype(np.float64)

# validation split from REAL training data — used only for hyperparameter tuning
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20,
    random_state=RANDOM_SEED, stratify=y_train_full)

_log('')
_log('── SECTION 3 : DATA LOADED ────────────────────────────────')
_log(f'  Features used            : {len(selected_features)}')
_log(f'  Real train (strategy A)  : {X_train_full.shape[0]:,} rows')
_log(f'    -> tuning split: train={X_tr.shape[0]:,}  val={X_val.shape[0]:,}')
_log(f'  SMOTE train (strategy B) : {X_smote.shape[0]:,} rows')
_log(f'  Test (held out, shared)  : {X_test.shape[0]:,} rows')
_log(f'  Real-train class balance : BENIGN={int((y_train_full==0).sum()):,}  '
     f'ATTACK={int((y_train_full==1).sum()):,}')
_log(f'  SMOTE-train class balance: BENIGN={int((y_smote==0).sum()):,}  '
     f'ATTACK={int((y_smote==1).sum()):,}')
_log(f'  Test class balance       : BENIGN={int((y_test==0).sum()):,}  '
     f'ATTACK={int((y_test==1).sum()):,}')
print('\nBoth training strategies loaded — shapes verified against Section 1.')

## 4. Logistic Regression — From Scratch (NumPy)
No `sklearn.linear_model`. We implement the whole thing:

**The model.** For features $x$ and weights $w$, bias $b$:
$$\hat{y} = \sigma(x \cdot w + b), \qquad \sigma(z) = \frac{1}{1+e^{-z}}$$

**The loss.** Binary cross-entropy with **L2 regularisation** (strength $\lambda$):
$$\mathcal{L} = -\frac{1}{n}\sum \big[y\log\hat{y} + (1-y)\log(1-\hat{y})\big] + \frac{\lambda}{2n}\lVert w\rVert^2$$

**The optimiser.** **Mini-batch gradient descent** — the data is shuffled each epoch and processed in batches of `batch_size`. Gradients:
$$\frac{\partial\mathcal{L}}{\partial w} = \frac{1}{m}X^\top(\hat{y}-y) + \frac{\lambda}{m}w, \qquad \frac{\partial\mathcal{L}}{\partial b} = \frac{1}{m}\sum(\hat{y}-y)$$

**Class imbalance.** The training set is ~83% BENIGN. Instead of loading the 3.4M-row SMOTE file, we weight the loss by inverse class frequency (`class_weight=True`) — same effect, implemented from scratch, no extra data.

**Tunable hyperparameters:** `lr` (learning rate), `lambda_` (L2 strength), `batch_size`, `epochs`.

In [ ]:
class LogisticRegressionScratch:
    """Binary logistic regression — pure NumPy, mini-batch GD, L2 reg, optional class weighting."""

    def __init__(self, lr=0.1, lambda_=0.0, batch_size=4096, epochs=30,
                 class_weight=True, random_state=42, verbose=False):
        self.lr = lr
        self.lambda_ = lambda_
        self.batch_size = batch_size
        self.epochs = epochs
        self.class_weight = class_weight
        self.random_state = random_state
        self.verbose = verbose
        self.w = None
        self.b = 0.0
        self.history = {'train_loss': [], 'val_loss': []}

    @staticmethod
    def _sigmoid(z):
        # clip to avoid overflow in exp
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def _loss(self, X, y, sample_w):
        p = self._sigmoid(X @ self.w + self.b)
        eps = 1e-12
        bce = -(sample_w * (y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))).mean()
        l2  = (self.lambda_ / (2 * len(y))) * np.sum(self.w ** 2)
        return bce + l2

    def fit(self, X, y, X_val=None, y_val=None):
        rng = np.random.RandomState(self.random_state)
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0

        # per-sample weights for class imbalance
        if self.class_weight:
            n_pos, n_neg = (y == 1).sum(), (y == 0).sum()
            w_pos, w_neg = n / (2.0 * n_pos), n / (2.0 * n_neg)
            sw = np.where(y == 1, w_pos, w_neg)
        else:
            sw = np.ones(n)
        sw_val = np.ones(len(y_val)) if y_val is not None else None

        for epoch in range(self.epochs):
            idx = rng.permutation(n)
            for start in range(0, n, self.batch_size):
                bi = idx[start:start + self.batch_size]
                Xb, yb, swb = X[bi], y[bi], sw[bi]
                m = len(yb)
                p = self._sigmoid(Xb @ self.w + self.b)
                err = (p - yb) * swb
                grad_w = (Xb.T @ err) / m + (self.lambda_ / m) * self.w
                grad_b = err.sum() / m
                self.w -= self.lr * grad_w
                self.b -= self.lr * grad_b
            # record epoch losses
            self.history['train_loss'].append(self._loss(X, y, sw))
            if X_val is not None:
                self.history['val_loss'].append(self._loss(X_val, y_val, sw_val))
            if self.verbose:
                msg = f'  epoch {epoch+1:3}/{self.epochs}  train_loss={self.history["train_loss"][-1]:.4f}'
                if X_val is not None:
                    msg += f'  val_loss={self.history["val_loss"][-1]:.4f}'
                print(msg)
        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

print('LogisticRegressionScratch defined.')
print('  sigmoid + binary cross-entropy + L2  ·  mini-batch GD  ·  class weighting')

In [ ]:
# quick smoke test — one fit on default hyperparameters to confirm it learns
_t = time.time()
_smoke = LogisticRegressionScratch(lr=0.1, lambda_=0.0, batch_size=8192,
                                   epochs=20, verbose=True)
_smoke.fit(X_tr, y_tr, X_val, y_val)
_val_f1 = f1_score(y_val, _smoke.predict(X_val))
_log('')
_log('── SECTION 4 : FROM-SCRATCH LR — SMOKE TEST ───────────────')
_log(f'  Default hyperparams (lr=0.1, lambda=0, batch=8192, epochs=20)')
_log(f'  Validation F1 : {_val_f1:.4f}   (fit took {time.time()-_t:.1f}s)')
_log(f'  Loss went {_smoke.history["train_loss"][0]:.4f} -> {_smoke.history["train_loss"][-1]:.4f}'
     '  (decreasing = learning correctly)')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(_smoke.history['train_loss'], 'o-', label='train loss', color='#1976D2')
ax.plot(_smoke.history['val_loss'],  's-', label='val loss',   color='#F44336')
ax.set_xlabel('epoch'); ax.set_ylabel('BCE loss (weighted)')
ax.set_title('Smoke Test — Loss Curve (confirms the from-scratch model learns)')
ax.legend()
plt.tight_layout()
_savefig('02_smoke_test_loss.png', fig)
plt.show()

## 5. Hyperparameter Tuning — Grid Search
We grid-search over the three core hyperparameters and pick the combination with the best **validation F1** (F1, not accuracy — accuracy is misleading on imbalanced data).

| Hyperparameter | Grid | Why it matters |
|---|---|---|
| `lr` (learning rate) | 0.01, 0.1, 0.5 | too low = slow / underfits; too high = diverges |
| `lambda_` (L2 strength) | 0.0, 0.1, 1.0 | controls overfitting; larger = simpler model |
| `batch_size` | 4096, 16384 | smaller = noisier but more updates; larger = smoother / faster per epoch |

`epochs` is fixed at 30 for the search (enough to compare fairly), then the best config is retrained. Every fit's loss curve is kept so we can see *how* each config trained, not just its final score.

In [ ]:
param_grid = {
    'lr':         [0.01, 0.1, 0.5],
    'lambda_':    [0.0, 0.1, 1.0],
    'batch_size': [4096, 16384],
}
GRID_EPOCHS = 30

results   = []
histories = {}   # (lr, lambda_, batch_size) -> history dict

_log('')
_log('── SECTION 5 : GRID SEARCH ────────────────────────────────')
_log(f'  Grid: {len(param_grid["lr"])} lr x {len(param_grid["lambda_"])} lambda '
     f'x {len(param_grid["batch_size"])} batch = '
     f'{len(param_grid["lr"])*len(param_grid["lambda_"])*len(param_grid["batch_size"])} fits')

t_all = time.time()
for lr in param_grid['lr']:
    for lam in param_grid['lambda_']:
        for bs in param_grid['batch_size']:
            t0 = time.time()
            model = LogisticRegressionScratch(lr=lr, lambda_=lam, batch_size=bs,
                                              epochs=GRID_EPOCHS, class_weight=True,
                                              random_state=RANDOM_SEED)
            model.fit(X_tr, y_tr, X_val, y_val)
            yv_pred = model.predict(X_val)
            yv_prob = model.predict_proba(X_val)
            row = {
                'lr': lr, 'lambda_': lam, 'batch_size': bs,
                'val_f1':        f1_score(y_val, yv_pred),
                'val_precision': precision_score(y_val, yv_pred),
                'val_recall':    recall_score(y_val, yv_pred),
                'val_accuracy':  accuracy_score(y_val, yv_pred),
                'val_auc':       roc_auc_score(y_val, yv_prob),
                'final_loss':    model.history['train_loss'][-1],
                'fit_seconds':   time.time() - t0,
            }
            results.append(row)
            histories[(lr, lam, bs)] = model.history
            _log(f'  lr={lr:<5} lambda={lam:<4} batch={bs:<6} -> '
                 f'val_F1={row["val_f1"]:.4f}  AUC={row["val_auc"]:.4f}  ({row["fit_seconds"]:.1f}s)')

results_df = pd.DataFrame(results).sort_values('val_f1', ascending=False).reset_index(drop=True)
_log('')
_log(f'  Total grid-search time : {time.time()-t_all:.1f}s')
_log('  Results (sorted by validation F1):')
_log(results_df.to_string(index=False))
display(results_df)

best = results_df.iloc[0]
_log('')
_log(f'  BEST CONFIG: lr={best.lr}, lambda_={best.lambda_}, batch_size={int(best.batch_size)}')
_log(f'              val_F1={best.val_f1:.4f}  val_AUC={best.val_auc:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 12))

# ── (a) heatmap: val F1 over lr x lambda, for each batch size ──
for k, bs in enumerate(param_grid['batch_size']):
    ax = axes[0, k]
    sub = results_df[results_df['batch_size'] == bs]
    pivot = sub.pivot(index='lr', columns='lambda_', values='val_f1')
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu', ax=ax,
                cbar_kws={'label': 'validation F1'})
    ax.set_title(f'Validation F1  —  batch_size = {bs}', fontweight='bold')
    ax.set_xlabel('L2 lambda'); ax.set_ylabel('learning rate')

# ── (b) loss curves for every config ──
ax = axes[1, 0]
for (lr, lam, bs), hist in histories.items():
    ax.plot(hist['val_loss'], alpha=0.6, label=f'lr={lr},λ={lam},bs={bs}')
ax.set_xlabel('epoch'); ax.set_ylabel('validation BCE loss')
ax.set_title('Validation Loss Curves — every grid config', fontweight='bold')
ax.legend(fontsize=6, ncol=2)

# ── (c) metric trade-off for the configs ──
ax = axes[1, 1]
top = results_df.head(8).iloc[::-1]
ylabels = [f'lr={r.lr},λ={r.lambda_},bs={int(r.batch_size)}' for r in top.itertuples()]
yy = np.arange(len(top))
ax.barh(yy - 0.25, top['val_precision'], 0.25, label='precision', color='#1976D2')
ax.barh(yy,        top['val_recall'],    0.25, label='recall',    color='#F44336')
ax.barh(yy + 0.25, top['val_f1'],        0.25, label='F1',        color='#4CAF50')
ax.set_yticks(yy); ax.set_yticklabels(ylabels, fontsize=8)
ax.set_xlabel('score'); ax.set_xlim(0, 1)
ax.set_title('Precision / Recall / F1 — top 8 configs', fontweight='bold')
ax.legend()

plt.suptitle('Hyperparameter Grid Search — From-Scratch Logistic Regression',
             fontsize=15, fontweight='bold')
plt.tight_layout()
_savefig('03_grid_search.png', fig)
plt.show()

## 6. Final Models — Train Both Strategies & Compare on Test
We take the **best hyperparameter config** from the grid search and train it **two ways**:

| | Strategy A — Class Weighting | Strategy B — SMOTE |
|---|---|---|
| Training data | `train_selected` (real, 2.06M rows) | `train_binary_smote_selected` (balanced, ~3.4M rows) |
| Imbalance handling | inverse-frequency loss weights | synthetic minority oversampling |
| `class_weight` flag | `True` | `False` (data already balanced) |

Both are evaluated **once** on the same held-out test set (real-world distribution). This answers the real question: *does synthesising minority samples actually beat simply re-weighting the loss?*

In [ ]:
best_lr, best_lam, best_bs = float(best.lr), float(best.lambda_), int(best.batch_size)
FINAL_EPOCHS = 60   # more epochs for the final fits

def evaluate(model, X, y):
    prob = model.predict_proba(X)
    pred = model.predict(X)
    return {
        'accuracy':  accuracy_score(y, pred),
        'precision': precision_score(y, pred),
        'recall':    recall_score(y, pred),
        'f1':        f1_score(y, pred),
        'roc_auc':   roc_auc_score(y, prob),
    }, pred, prob

_log('')
_log('-- SECTION 6 : FINAL MODELS -- STRATEGY A vs B --')
_log(f'  Best config (from grid search): lr={best_lr}, lambda_={best_lam}, '
     f'batch_size={best_bs}, epochs={FINAL_EPOCHS}')

# ---- Strategy A : real data + class weighting ----
_log('')
_log('  [A] Class weighting -- training on real-distribution data ...')
t0 = time.time()
model_A = LogisticRegressionScratch(lr=best_lr, lambda_=best_lam, batch_size=best_bs,
                                    epochs=FINAL_EPOCHS, class_weight=True,
                                    random_state=RANDOM_SEED)
model_A.fit(X_train_full, y_train_full, X_test, y_test)
metrics_A, pred_A, prob_A = evaluate(model_A, X_test, y_test)
_log(f'      done in {time.time()-t0:.1f}s')

# ---- Strategy B : SMOTE data, no class weighting ----
_log('  [B] SMOTE -- training on synthetically balanced data ...')
t0 = time.time()
model_B = LogisticRegressionScratch(lr=best_lr, lambda_=best_lam, batch_size=best_bs,
                                    epochs=FINAL_EPOCHS, class_weight=False,
                                    random_state=RANDOM_SEED)
model_B.fit(X_smote, y_smote, X_test, y_test)
metrics_B, pred_B, prob_B = evaluate(model_B, X_test, y_test)
_log(f'      done in {time.time()-t0:.1f}s')

# ---- comparison table ----
cmp_df = pd.DataFrame({'A_class_weight': metrics_A, 'B_smote': metrics_B})
cmp_df['difference'] = cmp_df['B_smote'] - cmp_df['A_class_weight']
cmp_df['winner'] = np.where(cmp_df['difference'].abs() < 1e-4, 'tie',
                            np.where(cmp_df['difference'] > 0, 'B_smote', 'A_class_weight'))

_log('')
_log('  TEST-SET COMPARISON (real-world distribution):')
_log(cmp_df.to_string())
display(cmp_df)

# overall winner by F1
f1_A = metrics_A['f1']
f1_B = metrics_B['f1']
overall = 'B (SMOTE)' if f1_B > f1_A else 'A (class weighting)'
_log('')
_log(f'  Overall winner by F1 : {overall}')
_log(f'    A class-weight F1 = {f1_A:.4f}  |  B SMOTE F1 = {f1_B:.4f}')
print(f'\nWinner by test F1: {overall}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 13))

# ---- (a) confusion matrix -- Strategy A ----
cm_A = confusion_matrix(y_test, pred_A)
sns.heatmap(cm_A, annot=True, fmt=',d', cmap='Blues', ax=axes[0, 0],
            xticklabels=['BENIGN', 'ATTACK'], yticklabels=['BENIGN', 'ATTACK'])
axes[0, 0].set_title('A -- Class Weighting\nConfusion Matrix', fontweight='bold')
axes[0, 0].set_xlabel('Predicted'); axes[0, 0].set_ylabel('Actual')

# ---- (b) confusion matrix -- Strategy B ----
cm_B = confusion_matrix(y_test, pred_B)
sns.heatmap(cm_B, annot=True, fmt=',d', cmap='Greens', ax=axes[0, 1],
            xticklabels=['BENIGN', 'ATTACK'], yticklabels=['BENIGN', 'ATTACK'])
axes[0, 1].set_title('B -- SMOTE\nConfusion Matrix', fontweight='bold')
axes[0, 1].set_xlabel('Predicted'); axes[0, 1].set_ylabel('Actual')

# ---- (c) ROC curves -- both on one axis ----
ax = axes[0, 2]
fpr_A, tpr_A, _ = roc_curve(y_test, prob_A)
fpr_B, tpr_B, _ = roc_curve(y_test, prob_B)
auc_A = metrics_A['roc_auc']
auc_B = metrics_B['roc_auc']
ax.plot(fpr_A, tpr_A, color='#1976D2', lw=2, label=f'A class-weight (AUC={auc_A:.4f})')
ax.plot(fpr_B, tpr_B, color='#4CAF50', lw=2, label=f'B SMOTE (AUC={auc_B:.4f})')
ax.plot([0, 1], [0, 1], '--', color='gray', lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve -- A vs B', fontweight='bold')
ax.legend(loc='lower right')

# ---- (d) grouped metric bars ----
ax = axes[1, 0]
metric_names = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
xx = np.arange(len(metric_names))
ax.bar(xx - 0.2, [metrics_A[m] for m in metric_names], 0.4,
       label='A class-weight', color='#1976D2')
ax.bar(xx + 0.2, [metrics_B[m] for m in metric_names], 0.4,
       label='B SMOTE', color='#4CAF50')
ax.set_xticks(xx); ax.set_xticklabels(metric_names, rotation=20)
ax.set_ylim(0, 1.12)
ax.set_title('Test Metrics -- A vs B', fontweight='bold')
ax.legend()
for i, m in enumerate(metric_names):
    ax.text(i - 0.2, metrics_A[m], f'{metrics_A[m]:.3f}', ha='center', va='bottom', fontsize=7)
    ax.text(i + 0.2, metrics_B[m], f'{metrics_B[m]:.3f}', ha='center', va='bottom', fontsize=7)

# ---- (e) PR curves -- both ----
ax = axes[1, 1]
prec_A, rec_A, _ = precision_recall_curve(y_test, prob_A)
prec_B, rec_B, _ = precision_recall_curve(y_test, prob_B)
ax.plot(rec_A, prec_A, color='#1976D2', lw=2, label='A class-weight')
ax.plot(rec_B, prec_B, color='#4CAF50', lw=2, label='B SMOTE')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve -- A vs B', fontweight='bold')
ax.legend(loc='lower left')

# ---- (f) loss curves -- both ----
ax = axes[1, 2]
ax.plot(model_A.history['train_loss'], color='#1976D2', lw=2, label='A train')
ax.plot(model_A.history['val_loss'],   color='#1976D2', lw=1.2, ls='--', label='A test')
ax.plot(model_B.history['train_loss'], color='#4CAF50', lw=2, label='B train')
ax.plot(model_B.history['val_loss'],   color='#4CAF50', lw=1.2, ls='--', label='B test')
ax.set_xlabel('epoch'); ax.set_ylabel('BCE loss')
ax.set_title('Training Loss Curves -- A vs B', fontweight='bold')
ax.legend(fontsize=8)

plt.suptitle('SMOTE vs Class-Weighting -- From-Scratch Logistic Regression (Binary)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
_savefig('04_smote_vs_classweight.png', fig)
plt.show()

In [ ]:
# pick the winning model (higher test F1) as THE final model
if metrics_B['f1'] > metrics_A['f1']:
    final_model, final_metrics, final_strategy = model_B, metrics_B, 'B_smote'
    final_pred, final_prob = pred_B, prob_B
else:
    final_model, final_metrics, final_strategy = model_A, metrics_A, 'A_class_weight'
    final_pred, final_prob = pred_A, prob_A

final_f1 = final_metrics['f1']
_log('')
_log(f'  Selected final model : strategy {final_strategy}  (test F1 = {final_f1:.4f})')

# learned weights of the winning model -- which features drive the ATTACK prediction
coef = pd.Series(final_model.w, index=selected_features).sort_values()
fig, ax = plt.subplots(figsize=(10, max(8, len(coef) * 0.28)))
colors = ['#F44336' if c < 0 else '#1976D2' for c in coef]
coef.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_title(f'Learned Weights -- Winning Model ({final_strategy})\n'
             '(blue pushes toward ATTACK, red toward BENIGN)', fontweight='bold')
ax.set_xlabel('weight value')
ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
_savefig('05_learned_weights.png', fig)
plt.show()

_log('')
_log('  Top 10 features pushing toward ATTACK:')
_log(coef.sort_values(ascending=False).head(10).to_string())
_log('  Top 10 features pushing toward BENIGN:')
_log(coef.sort_values().head(10).to_string())

## 7. Save Model, Metrics & Report

In [ ]:
# save BOTH models' weights (pure NumPy — weights + bias + config)
p1 = os.path.join(OUT_DIR, 'logreg_scratch_binary_A_classweight.npz')
np.savez(p1, weights=model_A.w, bias=model_A.b,
         features=np.array(selected_features),
         lr=best_lr, lambda_=best_lam, batch_size=best_bs, epochs=FINAL_EPOCHS,
         strategy='A_class_weight')

p2 = os.path.join(OUT_DIR, 'logreg_scratch_binary_B_smote.npz')
np.savez(p2, weights=model_B.w, bias=model_B.b,
         features=np.array(selected_features),
         lr=best_lr, lambda_=best_lam, batch_size=best_bs, epochs=FINAL_EPOCHS,
         strategy='B_smote')

# grid-search results
p3 = os.path.join(OUT_DIR, 'gridsearch_results.csv')
results_df.to_csv(p3, index=False)

# A-vs-B comparison table
p4 = os.path.join(OUT_DIR, 'smote_vs_classweight_comparison.csv')
cmp_df.to_csv(p4)

# final metrics + which strategy won
p5 = os.path.join(OUT_DIR, 'final_metrics.json')
with open(p5, 'w') as f:
    json.dump({
        'model': 'LogisticRegressionScratch',
        'task': 'binary',
        'best_hyperparameters': {'lr': best_lr, 'lambda_': best_lam,
                                 'batch_size': best_bs, 'epochs': FINAL_EPOCHS},
        'strategy_A_class_weight': metrics_A,
        'strategy_B_smote':        metrics_B,
        'winning_strategy':        final_strategy,
        'winning_metrics':         final_metrics,
    }, f, indent=2)

_log('')
_log('── SECTION 7 : FILES SAVED ────────────────────────────────')
for p in [p1, p2, p3, p4, p5]:
    _log(f'  {os.path.basename(p)}')
    print(f'  {os.path.basename(p)}')

In [ ]:
_log('')
_log('=' * 70)
_log('SUMMARY  --  BINARY MODELLING (Logistic Regression from scratch)')
_log('=' * 70)
_log(f'  Features used   : {len(selected_features)}')
_log(f'  Test rows       : {X_test.shape[0]:,}  (real-world distribution)')
_log('')
_log('  Best hyperparameters (grid search on validation F1):')
_log(f'    learning rate : {best_lr}')
_log(f'    L2 lambda     : {best_lam}')
_log(f'    batch size    : {best_bs}')
_log('')
_log('  IMBALANCE STRATEGY COMPARISON (test set):')
_log(f'    {"metric":<12} {"A class-weight":>16} {"B SMOTE":>16}')
for m in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    _log(f'    {m:<12} {metrics_A[m]:>16.4f} {metrics_B[m]:>16.4f}')
_log('')
_log(f'  WINNING STRATEGY  : {final_strategy}  (by test F1)')
_log(f'  Final test F1     : {final_metrics["f1"]:.4f}')
_log(f'  Final test ROC-AUC: {final_metrics["roc_auc"]:.4f}')
_log('')
_log('  This is scratch-model #1. Next: a second from-scratch model (MLP / neural net),')
_log('  library baselines (RF / XGBoost), then the multi-class task.')

FIGURE_INDEX = [
    ('01_data_journey.png',          'Full pipeline: rows & features at every stage'),
    ('02_smoke_test_loss.png',       'From-scratch model learns (loss decreasing)'),
    ('03_grid_search.png',           'Grid search heatmaps + loss curves + metric trade-off'),
    ('04_smote_vs_classweight.png',  'SMOTE vs class-weighting -- full side-by-side'),
    ('05_learned_weights.png',       'Winning model -- learned weights / feature contributions'),
]
_log('')
_log('  Figures:')
for fname, desc in FIGURE_INDEX:
    _log(f'    {fname:<32} {desc}')

write_report()

f1_A_final = metrics_A['f1']
f1_B_final = metrics_B['f1']
print('\n' + '=' * 55)
print('BINARY MODELLING COMPLETE -- Logistic Regression (scratch)')
print('=' * 55)
print(f'  Strategy A (class-weight) F1 : {f1_A_final:.4f}')
print(f'  Strategy B (SMOTE)        F1 : {f1_B_final:.4f}')
print(f'  Winner : {final_strategy}')
print(f'  Report  -> {OUT_DIR}/Modeling_Binary_Report.txt')
print(f'  Figures -> {FIGURES_DIR}/  ({len(FIGURE_INDEX)} figures)')